# Member 1 — Random Forest
**Algorithm:** Random Forest Classifier (Supervised)  
**Dataset:** NSL-KDD (`nsl_kdd_dataset.csv`) — 41 pre-scaled features  
**Training data:** SMOTE-balanced (from shared preprocessing)

### Hyperparameter Tuning Strategy
- **Phase 1:** `RandomizedSearchCV` (wide search, 5-fold CV, 50 iterations) to narrow search space
- **Phase 2:** `GridSearchCV` (fine-grained search around best params, 5-fold CV)
- Scoring metric: **F1** (best for imbalanced detection tasks)
- Final model trained on full SMOTE-balanced training set with best params

## 1. Import Libraries

In [ ]:
import pickle, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve
)
from scipy.stats import randint

print('Libraries loaded.')

## 2. Load Preprocessed Data

In [ ]:
with open('../data/processed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']        # SMOTE-balanced
X_test       = data['X_test']         # real-world distribution
y_train      = data['y_train']
y_test       = data['y_test']
feature_cols = data['feature_cols']

print(f'X_train (SMOTE)  : {X_train.shape} | normal={np.sum(y_train==0)}, attack={np.sum(y_train==1)}')
print(f'X_test  (real)   : {X_test.shape}  | normal={np.sum(y_test==0)}, attack={np.sum(y_test==1)}')

## 3. Phase 1 — RandomizedSearchCV (Wide Search)

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators'     : randint(100, 600),
    'max_depth'        : [None, 10, 20, 30, 40],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf' : randint(1, 10),
    'max_features'     : ['sqrt', 'log2', 0.3, 0.5],
    'bootstrap'        : [True, False],
}

rf_rand = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_rand.fit(X_train, y_train)

print(f'\nBest params (random): {rf_rand.best_params_}')
print(f'Best CV F1 (random) : {rf_rand.best_score_:.4f}')

## 4. Phase 2 — GridSearchCV (Fine-Grained Search)

In [ ]:
bp = rf_rand.best_params_

# Build grid around the best random search result
def neighbor_ints(val, step, low=1, n=3):
    return sorted(set([max(low, val-step*i) for i in range(n)] +
                      [val+step*i for i in range(n)]))

param_grid = {
    'n_estimators'     : neighbor_ints(bp['n_estimators'], 50, low=50),
    'max_depth'        : [bp['max_depth'],
                          None if bp['max_depth'] is None else bp['max_depth']+5],
    'min_samples_split': neighbor_ints(bp['min_samples_split'], 2, low=2),
    'min_samples_leaf' : neighbor_ints(bp['min_samples_leaf'], 1, low=1),
    'max_features'     : [bp['max_features']],
    'bootstrap'        : [bp['bootstrap']],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train)

best_params = rf_grid.best_params_
print(f'\nBest params (grid) : {best_params}')
print(f'Best CV F1 (grid)  : {rf_grid.best_score_:.4f}')

## 5. Train Final Model with Best Parameters

In [ ]:
rf_final = RandomForestClassifier(
    **best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X_train, y_train)
print('Final Random Forest trained.')
print(f'Parameters: {best_params}')

## 6. 5-Fold Cross-Validation on Training Set

In [ ]:
cv_f1  = cross_val_score(rf_final, X_train, y_train, cv=cv5, scoring='f1',       n_jobs=-1)
cv_acc = cross_val_score(rf_final, X_train, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)
cv_auc = cross_val_score(rf_final, X_train, y_train, cv=cv5, scoring='roc_auc',  n_jobs=-1)

print('=== 5-Fold Cross-Validation (on SMOTE training set) ===')
print(f'F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}   | per fold: {np.round(cv_f1, 4)}')
print(f'Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}  | per fold: {np.round(cv_acc, 4)}')
print(f'ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}  | per fold: {np.round(cv_auc, 4)}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, scores, name, color in zip(
    axes,
    [cv_f1, cv_acc, cv_auc],
    ['F1 Score', 'Accuracy', 'ROC-AUC'],
    ['steelblue', 'darkorange', 'green']
):
    ax.plot(range(1,6), scores, 'o-', color=color, lw=2, markersize=7)
    ax.axhline(scores.mean(), linestyle='--', color=color, alpha=0.5,
               label=f'Mean={scores.mean():.4f}')
    ax.set_title(f'CV {name}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(name)
    ax.legend(fontsize=9)
    ax.set_xticks(range(1,6))
plt.suptitle('Random Forest — 5-Fold CV Scores', fontsize=13)
plt.tight_layout()
plt.savefig('cv_scores_rf.png', dpi=150)
plt.show()

## 7. Test Set Evaluation

In [ ]:
y_pred      = rf_final.predict(X_test)
y_pred_prob = rf_final.predict_proba(X_test)[:, 1]

print('=== Test Set Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal','Attack']))

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_pred_prob)

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')

## 8. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal','Attack']).plot(
    cmap='Greens', ax=axes[0])
axes[0].set_title('Random Forest — Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='green', lw=2,
             label=f'RF (AUC={roc_auc:.4f})')
axes[1].plot([0,1],[0,1],'--', color='navy', lw=1, label='Random')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('Random Forest — ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('cm_roc_rf.png', dpi=150)
plt.show()

## 9. Feature Importance

In [ ]:
feat_df = pd.DataFrame({'Feature': feature_cols,
                        'Importance': rf_final.feature_importances_})\
            .sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feat_df.head(15), x='Importance', y='Feature', palette='Greens_r')
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150)
plt.show()

## 10. Save Results

In [ ]:
os.makedirs('../comparison', exist_ok=True)
results = pd.DataFrame([{
    'Algorithm':'Random Forest','Type':'Supervised',
    'Accuracy':round(accuracy,4),'Precision':round(precision,4),
    'Recall':round(recall,4),'F1 Score':round(f1,4),'ROC-AUC':round(roc_auc,4),
    'CV F1 Mean':round(cv_f1.mean(),4),'CV F1 Std':round(cv_f1.std(),4)
}])
print(results.to_string(index=False))
results.to_csv('../comparison/results_random_forest.csv', index=False)
print('\nSaved: ../comparison/results_random_forest.csv')